# Xenium Run 1 – 02 Cell-type and Tg-subtype annotation

**Input:** `20251009_Run1_reSeg_QC.h5ad` (notebook 01)
**Output:**
* `ALLcells_run1.h5ad` – all annotated cells (`compartment_annotation_2`, `Tg_refined`) → cell-type composition
* `20260417_TBcells_subannotated.h5ad` – Tg + T + B cells (`Tg_annotation_full`) → T/B-border inference

Steps
1. Exclude section `6:Day3_0388` (QC)
2. Major cell types: module score + per-type threshold (33rd percentile)
3. Tg T cells: QC (MAD / Tukey / 2D-doublet filter) and module-score filter (top 25 %)
4. Tg subtypes, two-step annotation: lineage (Tfh vs Th1) → maturation (pre vs mature)
5. Refined annotation of all cells (extended marker set, GC B cells = top 25 % Aicda score)
6. Merge Tg subtypes back and export

Next notebook: `Xenium_03_TB_border_combined.ipynb`

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats, sparse

In [ ]:
# ------------------------------------------------------------
# Paths relative to this folder (adapt BASE_DIR if your data live elsewhere)
# ------------------------------------------------------------
BASE_DIR = ".."
RAW_DIR  = f"{BASE_DIR}/01_Raw"
DATA_DIR = f"{BASE_DIR}/02_ProcessedData"

QC_IN      = f"{DATA_DIR}/20251009_Run1_reSeg_QC.h5ad"          # from notebook 01
ALL_OUT    = f"{DATA_DIR}/ALLcells_run1.h5ad"                   # all annotated cells
TB_OUT     = f"{DATA_DIR}/20260417_TBcells_subannotated.h5ad"   # Tg + T + B cells

EXCLUDE_SECTIONS = ["6:Day3_0388"]   # excluded after QC

## 2. Load data and exclude sections

In [ ]:
adata = sc.read(QC_IN)
adata = adata[~adata.obs["timepoint"].isin(EXCLUDE_SECTIONS)].copy()
adata

Colours, sample order and marker genes for the major cell types.

In [ ]:
ln_major_colors = {
    "B_cells": "#4A90E2",        # soft blue
    "T_cells": "#50E3C2",        # gentle green
    "Tg_T_cells": "#D0021B",     # magenta / highlight
    "Macrophages": "#F5A623",    # warm gold
    "Stromal_cells": "#9B9B9B",  # neutral gray-violet
    "Lymphatic_EC": "#2E8B57",    # teal-cyan,
    "Tregs": "#7B6FD0",          # muted violet
    "cDC1": "#084594",           # deep navy blue (clearly blue)
    "cDC2": "#6B8E23",           # olive / yellow-green (clearly green)
    "GC_B_cells": "#6A3D9A",      # deep purple
    "Neutrophils": "#636363",
    "NA": "white"
}
ordered_timepoints = [
    "10:Day0_0382",
    "2:Day2_0386", "8:Day2_0385", "9:Day2_0385",
    "4:Day3_0388", "6:Day3_0388", "7:Day3_0388",
    "3:Day4_0391", "5:Day4_0391",
    "1:Day6_0394"
]

In [ ]:
ln_major_markers = {
    "B_cells": ["Cd79a","Cr2", "Cd38", "Cxcr5"],
    "T_cells": ["Cd3d", "Ccr7", "Cd4", "Cd8a"],
    "Tg_T_cells": ["ENA_Y11102_Y11102.1", "ENA_Y11103_Y11103.1", "CD45.1", "Cd3d", "Cd4"],
    "Macrophages": ["Cd68", "Sirpa"],
    "Stromal_cells": ["Timp3", "Cfh", "Cd21", "Cxcl13", "Ccl21"],
    "Lymphatic_EC": ["Lyve1", "Prox1", "Plvap"]
}

### Composition with the first annotation (notebook 01)

In [ ]:
# Remove entries with "Unknown" timepoint
adata_filtered = adata[adata.obs["timepoint"] != "Unknown"].copy()

# Compute counts per timepoint and cell type
cell_fractions = (
    adata_filtered.obs.groupby(["timepoint", "compartment_annotation"])
    .size()
    .reset_index(name="count")
)

# Normalize per timepoint → percentage
cell_fractions["fraction"] = (
    cell_fractions.groupby("timepoint")["count"].transform(lambda x: x / x.sum() * 100)
)

# Ensure consistent ordering
cell_fractions["timepoint"] = pd.Categorical(
    cell_fractions["timepoint"],
    categories=ordered_timepoints,
    ordered=True
)
cell_fractions

In [ ]:
pivot_df = cell_fractions.pivot_table(index='timepoint', columns='compartment_annotation', values='fraction', fill_value=0)

# Plot stacked barplot
pivot_df.plot(kind='bar', stacked=True, color=[ln_major_colors.get(c, "#7f7f7f") for c in pivot_df.columns], figsize=(8,3))
#pivot_df.plot(kind='bar', stacked=True,  figsize=(8,3))
plt.ylabel("Count")
plt.xlabel("Timepoint")
plt.title("Stacked Barplot of Cell Fractions per Superclass")
plt.xticks(rotation=45)
plt.legend(title="Superclass", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Major cell types – module score with per-type threshold

Every cell gets the type with the highest module score, but only if this score is at least the
33rd percentile of that score across all cells; otherwise the cell is left unassigned (`NA`).
Cells outside the sections (`Unknown`) are removed.

In [ ]:
# 1. Compute gene set scores
for comp, genes in ln_major_markers.items():
    sc.tl.score_genes(adata, gene_list=genes, score_name=f"{comp}_score")

# 2. Define score columns
compartments = list(ln_major_markers.keys())
score_cols = [f"{c}_score" for c in compartments]
scores = adata.obs[score_cols].copy()

# 3. Per-cell type thresholds: top 25% of cells retained
thresholds = scores.quantile(0.33)

# 4. Best label and score per cell
adata.obs["best_compartment_score_col"] = scores.idxmax(axis=1)
adata.obs["best_compartment_score"] = scores.max(axis=1)

# 5. Map threshold of the winning compartment to each cell
adata.obs["best_compartment_threshold"] = adata.obs["best_compartment_score_col"].map(thresholds)

# 6. Final annotation
adata.obs["compartment_annotation_2"] = np.where(
    adata.obs["best_compartment_score"] >= adata.obs["best_compartment_threshold"],
    adata.obs["best_compartment_score_col"].str.replace("_score", "", regex=False),
    pd.NA
)

In [ ]:
# Remove entries with "Unknown" timepoint
adata = adata[adata.obs["timepoint"] != "Unknown"].copy()

# Compute counts per timepoint and cell type
cell_fractions = (
    adata.obs.groupby(["timepoint", "compartment_annotation_2"])
    .size()
    .reset_index(name="count")
)

# Normalize per timepoint → percentage
cell_fractions["fraction"] = (
    cell_fractions.groupby("timepoint")["count"].transform(lambda x: x / x.sum() * 100)
)

# Ensure consistent ordering
cell_fractions["timepoint"] = pd.Categorical(
    cell_fractions["timepoint"],
    categories=ordered_timepoints,
    ordered=True
)
cell_fractions

In [ ]:
pivot_df = cell_fractions.pivot_table(index='timepoint', columns='compartment_annotation_2', values='fraction', fill_value=0)

# Plot stacked barplot
pivot_df.plot(kind='bar', stacked=True, color=[ln_major_colors.get(c, "#7f7f7f") for c in pivot_df.columns], figsize=(8,3))
#pivot_df.plot(kind='bar', stacked=True,  figsize=(8,3))
plt.ylabel("Count")
plt.xlabel("Timepoint")
plt.title("Stacked Barplot of Cell Fractions per Superclass")
plt.xticks(rotation=45)
plt.legend(title="Superclass", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Tg T cells

### 4.1 QC
* transcripts (log) and detected genes: median ± 3 MAD
* cell area: Tukey fences (1.5 × IQR)
* 2D doublets: cells with both counts and area ≥ 99th percentile

In [ ]:
adata_Tg_Tcells = adata[adata.obs['compartment_annotation_2'] == 'Tg_T_cells'].copy()

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata_Tg_Tcells.obs["total_counts"],
    kde=False,
    ax=axs[0],
)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata_Tg_Tcells.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1],
)


axs[2].set_title("Area of segmented cells")
sns.histplot(
    adata_Tg_Tcells.obs["cell_area"],
    kde=False,
    ax=axs[2],
)

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata_Tg_Tcells.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3],
)

In [ ]:
# Helper: robust bounds using MAD
def mad_bounds(x, k_low=3, k_high=3, log=False):
    x = np.asarray(x, dtype=float)
    x = np.log1p(x) if log else x
    med = np.nanmedian(x)
    mad = stats.median_abs_deviation(x, nan_policy='omit', scale='normal')
    lower = med - k_low * mad
    upper = med + k_high * mad
    return (np.expm1(lower) if log else lower,
            np.expm1(upper) if log else upper)

# 2) counts & genes: drop low and high outliers
lc_lo, lc_hi = mad_bounds(adata_Tg_Tcells.obs['total_counts'], 3, 3, log=True)
lg_lo, lg_hi = mad_bounds(adata_Tg_Tcells.obs['n_genes_by_counts'], 3, 3, log=False)

m_counts = (adata_Tg_Tcells.obs['total_counts'] >= lc_lo) & (adata_Tg_Tcells.obs['total_counts'] <= lc_hi)
m_genes  = (adata_Tg_Tcells.obs['n_genes_by_counts'] >= lg_lo) & (adata_Tg_Tcells.obs['n_genes_by_counts'] <= lg_hi)

# 3) cell area: Tukey bounds
q1, q3 = np.nanpercentile(adata_Tg_Tcells.obs['cell_area'], [25, 75])
iqr = q3 - q1
area_lo = q1 - 1.5*iqr
area_hi = q3 + 1.5*iqr
m_area = (adata_Tg_Tcells.obs['cell_area'] >= area_lo) & (adata_Tg_Tcells.obs['cell_area'] <= area_hi)

# 4) nucleus ratio: keep properly segmented cells
#nr = adata_Tg_Tcells.obs['nucleus_ratio']
#m_nuc = (nr >= 0.30) & (nr <= 0.90) & (~np.isclose(nr, 1.0))

# Optional extra guard against doublets: very large area & very high counts simultaneously
p99_counts = np.nanpercentile(adata_Tg_Tcells.obs['total_counts'], 99)
p99_area   = np.nanpercentile(adata_Tg_Tcells.obs['cell_area'], 99)
m_not_doublet2d = ~((adata_Tg_Tcells.obs['total_counts'] >= p99_counts) & (adata_Tg_Tcells.obs['cell_area'] >= p99_area))

# Combine all filters
mask = m_counts & m_genes & m_area & m_not_doublet2d
adata_tg_filt = adata_Tg_Tcells[mask].copy()

print(adata_Tg_Tcells.shape, "→", adata_tg_filt.shape)

### 4.2 Keep Tg T cells with a high Tg module score (top 25 %)

In [ ]:
plt.figure(figsize=(2, 5))
sns.boxplot(
    data=adata_tg_filt.obs,
    x="compartment_annotation_2",
    y="Tg_T_cells_score",
    showcaps=True,
    fliersize=1,
    linewidth=1,
    color="skyblue"
)

plt.xticks(rotation=45, ha="right")
plt.title("Distribution of transgenic T cell module scores per Tg_annotation")
plt.tight_layout()


plt.show()

In [ ]:
threshold = adata_tg_filt.obs["Tg_T_cells_score"].quantile(0.75)  # 75th percentile cutoff
adata_Tg_Tcells_filt = adata_tg_filt[
    adata_tg_filt.obs["Tg_T_cells_score"] > threshold
].copy()

print(f"Filtered from {adata_tg_filt.n_obs} to {adata_Tg_Tcells_filt.n_obs} cells.")

In [ ]:
# Count number of Tg cells per timepoint

adata_Tg_Tcells_filt.obs["timepoint"].value_counts().sort_index()

### 4.3 Tg subtypes – two-step annotation

1. **Lineage:** Tfh score (Cxcr5, Pdcd1, Il21, Bcl6, Icos) vs Th1 score (Cxcr3, Tbx21, S100a4); higher score wins.
2. **Maturation within the lineage:** mature score − early score
   (Tfh: Cxcr5, Pdcd1, Il21, Bcl6 vs Zbtb20, Arid5b, Tcf7, Ccr7, Bach2;
   Th1: Cxcr3, Tbx21, S100a4 vs Lyar, Fabp5, Cenpw, Il2ra). > 0 → mature, otherwise pre-.

Result: `Tg_pre_Tfh`, `Tg_Tfh`, `Tg_pre_Th1`, `Tg_Th1` in `Tg_annotation`.

In [ ]:
# ------------------------------------------------------------
# 1. Define marker panels
# ------------------------------------------------------------

# Lineage-defining markers
tg_lineage_markers = {
    "Tfh_lineage": [
        "Cxcr5",
        "Pdcd1",
        "Il21",
        "Bcl6",
        "Icos"
    ],
    "Th1_lineage": [
        "Cxcr3",
        "Tbx21",
        "S100a4"
    ]
}

# Maturation markers within each lineage
tg_maturation_markers = {
    "Tfh_early": [
        "Zbtb20",
        "Arid5b",
        "Tcf7",
        "Ccr7",
        "Bach2"
    ],
    "Tfh_mature": [
        "Cxcr5",
        "Pdcd1",
        "Il21",
        "Bcl6"
    ],

    "Th1_early": [
        "Lyar",
        "Fabp5",
        "Cenpw",
        "Il2ra"
    ],
    "Th1_mature": [
        "Cxcr3",
        "Tbx21",
        "S100a4"
    ]
}


# ------------------------------------------------------------
# 2. Compute lineage scores
# ------------------------------------------------------------

for lineage, genes in tg_lineage_markers.items():

    genes_present = [
        g for g in genes
        if g in adata_Tg_Tcells_filt.var_names
    ]

    sc.tl.score_genes(
        adata_Tg_Tcells_filt,
        gene_list=genes_present,
        score_name=f"{lineage}_score"
    )


# ------------------------------------------------------------
# 3. Classify Tfh-like vs Th1-like
# ------------------------------------------------------------

lineage_score_cols = [
    "Tfh_lineage_score",
    "Th1_lineage_score"
]

adata_Tg_Tcells_filt.obs["Tg_lineage"] = (
    adata_Tg_Tcells_filt.obs[lineage_score_cols]
    .idxmax(axis=1)
    .str.replace("_lineage_score", "", regex=False)
)

# Rename to nicer labels
adata_Tg_Tcells_filt.obs["Tg_lineage"] = (
    adata_Tg_Tcells_filt.obs["Tg_lineage"]
    .replace({
        "Tfh": "Tfh",
        "Th1": "Th1"
    })
)


# ------------------------------------------------------------
# 4. Calculate lineage margin
# ------------------------------------------------------------
# Useful for identifying ambiguous cells

adata_Tg_Tcells_filt.obs["Tg_lineage_margin"] = (
    adata_Tg_Tcells_filt.obs["Tfh_lineage_score"]
    - adata_Tg_Tcells_filt.obs["Th1_lineage_score"]
)

# Absolute margin = confidence in lineage assignment
adata_Tg_Tcells_filt.obs["Tg_lineage_confidence"] = (
    adata_Tg_Tcells_filt.obs["Tg_lineage_margin"].abs()
)


# ------------------------------------------------------------
# 5. Compute maturation scores
# ------------------------------------------------------------

for state, genes in tg_maturation_markers.items():

    genes_present = [
        g for g in genes
        if g in adata_Tg_Tcells_filt.var_names
    ]

    sc.tl.score_genes(
        adata_Tg_Tcells_filt,
        gene_list=genes_present,
        score_name=f"{state}_score"
    )


# ------------------------------------------------------------
# 6. Calculate continuous maturation scores
# ------------------------------------------------------------

# Positive = more mature Tfh-like
# Negative = more early/pre-Tfh-like
adata_Tg_Tcells_filt.obs["Tfh_maturation_score"] = (
    adata_Tg_Tcells_filt.obs["Tfh_mature_score"]
    - adata_Tg_Tcells_filt.obs["Tfh_early_score"]
)

# Positive = more mature Th1-like
# Negative = more early/pre-Th1-like
adata_Tg_Tcells_filt.obs["Th1_maturation_score"] = (
    adata_Tg_Tcells_filt.obs["Th1_mature_score"]
    - adata_Tg_Tcells_filt.obs["Th1_early_score"]
)


# ------------------------------------------------------------
# 7. Convert maturation score into 4 categorical states
# ------------------------------------------------------------

def assign_tg_state(row):

    if row["Tg_lineage"] == "Tfh":

        if row["Tfh_maturation_score"] > 0:
            return "Tg_Tfh"
        else:
            return "Tg_pre_Tfh"

    elif row["Tg_lineage"] == "Th1":

        if row["Th1_maturation_score"] > 0:
            return "Tg_Th1"
        else:
            return "Tg_pre_Th1"

    return "Unassigned"


adata_Tg_Tcells_filt.obs["Tg_annotation"] = (
    adata_Tg_Tcells_filt.obs.apply(
        assign_tg_state,
        axis=1
    )
)


# ------------------------------------------------------------
# 8. Set category order
# ------------------------------------------------------------

tg_order = [
    "Tg_pre_Tfh",
    "Tg_Tfh",
    "Tg_pre_Th1",
    "Tg_Th1"
]

adata_Tg_Tcells_filt.obs["Tg_annotation"] = pd.Categorical(
    adata_Tg_Tcells_filt.obs["Tg_annotation"],
    categories=tg_order,
    ordered=True
)


# ------------------------------------------------------------
# 9. Extract day from timepoint
# ------------------------------------------------------------

# Example:
# "1:Day6_0394" -> "Day6"

adata_Tg_Tcells_filt.obs["day"] = (
    adata_Tg_Tcells_filt.obs["timepoint"]
    .astype(str)
    .str.extract(r"(Day\d+)")[0]
)

day_order = [
    "Day0",
    "Day2",
    "Day3",
    "Day4",
    "Day6"
]

adata_Tg_Tcells_filt.obs["day"] = pd.Categorical(
    adata_Tg_Tcells_filt.obs["day"],
    categories=day_order,
    ordered=True
)


# ------------------------------------------------------------
# 10. Cell numbers and percentages per day
# ------------------------------------------------------------

tg_day_table = (
    adata_Tg_Tcells_filt.obs
    .groupby(
        ["day", "Tg_annotation"],
        observed=False
    )
    .size()
    .rename("n_cells")
    .reset_index()
)

tg_day_table["percentage_within_day"] = (
    tg_day_table
    .groupby("day", observed=False)["n_cells"]
    .transform(lambda x: x / x.sum() * 100)
)

tg_day_table["percentage_within_day"] = (
    tg_day_table["percentage_within_day"]
    .round(1)
)

display(tg_day_table)

### 4.4 Spatial overview and marker genes per Tg subtype

In [ ]:
# Make sure it's categorical
adata_Tg_Tcells_filt.obs["Tg_annotation"] = (
    adata_Tg_Tcells_filt.obs["Tg_annotation"].astype("category")
)

module_palette2 = {
    "Tg_pre_Tfh":  "#6baed6",
    "Tg_Tfh":      "#08519c",
    "Tg_pre_Th1":  "#fdae6b",
    "Tg_Th1":      "#a63603"
}

# Build color list in the exact order of categories
cats = adata_Tg_Tcells_filt.obs["Tg_annotation"].cat.categories
adata_Tg_Tcells_filt.uns["Tg_annotation_colors"] = [
    module_palette2.get(c, "#bdbdbd") for c in cats
]

sq.pl.spatial_scatter(
    adata_Tg_Tcells_filt,
    library_id="spatial",
    shape=None,
    color=["Tg_annotation"],
    wspace=0.4,
)

In [ ]:
sc.tl.rank_genes_groups(
    adata_Tg_Tcells_filt,
    groupby="Tg_annotation",
    method="wilcoxon",     # or 't-test' / 'logreg'
    n_genes=adata_Tg_Tcells.raw.shape[1] if adata_Tg_Tcells.raw is not None else None
)

In [ ]:
sc.pl.rank_genes_groups(adata_Tg_Tcells_filt, n_genes=50, sharey=False)

## 5. Refined annotation of all cells

Extended marker set (Tregs, cDC1, cDC2, GC B cells, neutrophils); highest module score wins.
All cells called Tg by this step are reset to `NA` and only the QC- and score-filtered Tg cells of
section 4 are kept as `Tg_T_cells`. GC B cells: only cells in the top 25 % of the Aicda score stay
GC B cells, the others become B cells. Unassigned (`NA`) cells are removed.

In [ ]:
ln_major_markers2  = {
    "Tregs": ["Cd3d","Cd4", "Foxp3", "Il2ra"],
    "cDC1": ["Xcr1", "Itgax", "Cd74", "Irf8"],
    "cDC2": ["Sirpa", "Itgax", "Cd74", "Il6"],
    #"GC_B_cells": ["Cd79a", "Cxcr5", "Aicda", "Bcl6"],
    "GC_B_cells": ["Aicda"],
    "B_cells": ["Cd79a","Cr2", "Cd38", "Cxcr5"],
    "T_cells": ["Cd3d", "Ccr7", "Cd4", "Cd8a"],
    "Tg_T_cells": ["ENA_Y11102_Y11102.1", "ENA_Y11103_Y11103.1", "CD45.1", "Cd3d", "Cd4"],
    "Macrophages": ["Cd68", "Sirpa"],
    "Stromal_cells": ["Timp3", "Cfh", "Cd21", "Cxcl13", "Ccl21"],
    "Lymphatic_EC": ["Lyve1", "Prox1", "Plvap"],
    "Neutrophils" :["Lst1", "Csf3r", "Pglyrp1", "Mmp9"]
}

In [ ]:
# 1. Compute gene set scores
for comp, genes in ln_major_markers2.items():
    sc.tl.score_genes(adata, gene_list=genes, score_name=f"{comp}_score")

# 2. Get compartment list
compartments = list(ln_major_markers2.keys())

# 3. Annotate each cell by its highest score
adata.obs["compartment_annotation_2"] = adata.obs[
    [f"{c}_score" for c in compartments]
].idxmax(axis=1).str.replace("_score", "")

In [ ]:
adata.obs["compartment_annotation_2"] = adata.obs["compartment_annotation_2"].astype(str)
# 1. get filtered Tg cells
filtered_cells = set(adata_Tg_Tcells_filt.obs_names)

# 2. identify Tg cells in original adata
is_tg_original = adata.obs["compartment_annotation_2"].str.startswith("Tg")

# 3. set ALL original Tg cells to NA first
adata.obs.loc[is_tg_original, "compartment_annotation_2"] = "NA"

# 4. overwrite with filtered Tg annotations
adata.obs.loc[
    adata.obs_names.isin(filtered_cells),
    "compartment_annotation_2"
] = adata_Tg_Tcells_filt.obs.loc[
    adata.obs_names.intersection(adata_Tg_Tcells_filt.obs_names),
    "compartment_annotation_2"
]

In [ ]:
adata.obs["compartment_annotation_2"].value_counts()

### GC B cells – keep top 25 % Aicda score

In [ ]:
adata_gc = adata[adata.obs['compartment_annotation_2'] == 'GC_B_cells'].copy()

In [ ]:
plt.figure(figsize=(2, 5))
sns.boxplot(
    data=adata_gc.obs,
    x="compartment_annotation_2",
    y="GC_B_cells_score",
    showcaps=True,
    fliersize=1,
    linewidth=1,
    color="skyblue"
)

plt.xticks(rotation=45, ha="right")
plt.title("Distribution of GC_B_cells module scores")
plt.tight_layout()


plt.show()

In [ ]:
print("GC_B_cells_score stats:")
print(adata_gc.obs['GC_B_cells_score'].describe())
print("\nTop 25% threshold:", adata_gc.obs['GC_B_cells_score'].quantile(0.75))
print("Max score:", adata_gc.obs['GC_B_cells_score'].max())

In [ ]:
#subset GC_B_cells to only HIGH Aicda scores
high_gc_threshold = adata_gc.obs['GC_B_cells_score'].quantile(0.75) # still high for 1.55
gc_mask = (adata.obs['compartment_annotation_2'] == 'GC_B_cells') & \
          (adata.obs['GC_B_cells_score'] > high_gc_threshold)

# Reassign low-confidence GC_B_cells
adata.obs.loc[~gc_mask & (adata.obs['compartment_annotation_2'] == 'GC_B_cells'),
              'compartment_annotation_2'] = 'B_cells' #(for the time being will change later)

print(f"High-confidence GC_B_cells: {gc_mask.sum()}")
print(adata.obs['compartment_annotation_2'].value_counts())

### Remove unassigned cells and plot the final annotation

In [ ]:
# convert literal "NA" strings to actual missing values
adata.obs["compartment_annotation_2"] = adata.obs["compartment_annotation_2"].replace("NA", pd.NA)

# now remove missing cells
adata = adata[adata.obs["compartment_annotation_2"].notna()].copy()

adata.obs["compartment_annotation_2"] = adata.obs["compartment_annotation_2"].astype("category")
adata.uns["compartment_annotation_2_colors"] = [
    ln_major_colors[c] for c in adata.obs["compartment_annotation_2"].cat.categories
]
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color=[
        "compartment_annotation_2",
    ],
    wspace=0.4,
    figsize=(20, 20),
    dpi=600,
    save="compartment_annotation_2_spatial_highres.png"
)

# vector version

plt.savefig(
    "compartment_annotation_2_spatial_highres.pdf",
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

### Composition per section (final annotation)

In [ ]:
# Remove entries with "Unknown" timepoint
adata_filtered = adata[adata.obs["timepoint"] != "Unknown"].copy()

# Compute counts per timepoint and cell type
cell_fractions = (
    adata_filtered.obs.groupby(["timepoint", "compartment_annotation_2"])
    .size()
    .reset_index(name="count")
)

# Normalize per timepoint → percentage
cell_fractions["fraction"] = (
    cell_fractions.groupby("timepoint")["count"].transform(lambda x: x / x.sum() * 100)
)

# Ensure consistent ordering
cell_fractions["timepoint"] = pd.Categorical(
    cell_fractions["timepoint"],
    categories=ordered_timepoints,
    ordered=True
)

In [ ]:
pivot_df = cell_fractions.pivot_table(index='timepoint', columns='compartment_annotation_2', values='fraction', fill_value=0)

# Plot stacked barplot
pivot_df.plot(kind='bar', stacked=True, color=[ln_major_colors.get(c, "#7f7f7f") for c in pivot_df.columns], figsize=(8,3))
#pivot_df.plot(kind='bar', stacked=True,  figsize=(8,3))
plt.ylabel("Count")
plt.xlabel("Timepoint")
plt.title("Stacked Barplot of Cell Fractions per Superclass")
plt.xticks(rotation=45)
plt.legend(title="Superclass", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Pivot table: timepoint vs GC_B_cells fraction
gcb_pivot = cell_fractions[cell_fractions['compartment_annotation_2'] == 'GC_B_cells'].pivot_table(
    index='timepoint',
    values='fraction',
    aggfunc='sum'
) # Convert to percent

print("GC_B_cells per timepoint:")
print(gcb_pivot)

## 6. Merge Tg subtypes into the full object and export

* `Tg_refined`: Tg subtype for Tg cells, `non_Tg` otherwise
* `nhood_annotation`: `compartment_annotation_2` with Tg T cells replaced by their subtype

In [ ]:
# pick your refined label column in adata_Tg_Tcells_filt
refined_key = "Tg_annotation"   # or "Tg_annotation_refined", etc.

# create a column in full adata with refined Tg labels (others = "non_Tg")
adata.obs["Tg_refined"] = "non_Tg"
adata.obs.loc[adata_Tg_Tcells_filt.obs_names, "Tg_refined"] = (
    adata_Tg_Tcells_filt.obs[refined_key].astype(str).values
)
adata.obs["Tg_refined"] = adata.obs["Tg_refined"].astype("category")

In [ ]:
ln_key = "compartment_annotation_2"
tg_key = "Tg_refined"

In [ ]:
adata.obs["nhood_annotation"] = adata.obs[ln_key].astype(str)

# identify Tg T cells in LN annotation
is_tg = adata.obs[ln_key] == "Tg_T_cells"

# overwrite only Tg_T_cells with refined labels
adata.obs.loc[is_tg, "nhood_annotation"] = (
    adata.obs.loc[is_tg, tg_key].astype(str)
)

adata.obs["nhood_annotation"] = adata.obs["nhood_annotation"].astype("category")

In [ ]:
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color=[
        "nhood_annotation",
    ],
    wspace=0.4,
)

### Save all annotated cells (input for composition analyses)

In [ ]:
adata.write(ALL_OUT)
print(adata.shape, '->', ALL_OUT)

### Tg + T + B cells (input for T/B-border inference)

In [ ]:
adata_TB = adata[adata.obs["compartment_annotation_2"].isin(["B_cells", "T_cells"])].copy()
print(adata_TB.shape)

In [ ]:
adata_TgT_B = ad.concat(
    [adata_Tg_Tcells_filt, adata_TB],
    join="outer",
    label="source",
    keys=["Tg_cells", "TB_cells"],
    fill_value=0
)
print(adata_TgT_B.shape)

In [ ]:
adata_TgT_B.obs["Tg_annotation_full"] = np.select(
    [
        adata_TgT_B.obs["compartment_annotation_2"] == "B_cells",
        adata_TgT_B.obs["compartment_annotation_2"] == "T_cells"
    ],
    [
        "B_cells",
        "T_cells"
    ],
    default=adata_TgT_B.obs["Tg_annotation"]
)

# Make it categorical
adata_TgT_B.obs["Tg_annotation_full"] = adata_TgT_B.obs["Tg_annotation_full"].astype("category")

# Quick sanity check
adata_TgT_B.obs["Tg_annotation_full"].value_counts()

In [ ]:
adata_TgT_B.write(TB_OUT)
print(adata_TgT_B.shape, '->', TB_OUT)